# Notebook 06 — Phylogenetic Trees: Distance Methods

**Module:** 06 — Genetics and Evolution  
**Notebook:** 06 of 12  
**Prerequisites:** Module 05 NB09 (UPGMA, JC distance)  
**Time estimate:** 90 minutes

> Module 05 NB09 introduced UPGMA and JC distance. This notebook adds Neighbour-Joining,
> bootstrapping, and the biological interpretation of tree topologies. NJ is the
> standard fast distance method and appears in every phylogenetics methods section.

---
## Step 1 — Motivation

UPGMA assumes a molecular clock (all lineages evolve at the same rate). Real data
often violates this. Neighbour-Joining (NJ) makes no such assumption — it finds the
tree minimising the total branch length without imposing a rate. NJ is faster than
maximum likelihood for large datasets and often used as a starting tree for ML.
Understanding its logic is essential for reading phylogenetics methods sections.

---
## Step 3 — Biological Background

**Distance-based phylogenetics: the pipeline**
1. Align sequences (MAFFT, MUSCLE, or manual for short sequences)
2. Compute pairwise distances (JC, Kimura 2-parameter, or more complex models)
3. Build tree from distance matrix (UPGMA or NJ)
4. Bootstrap: resample columns of the alignment, rebuild trees, record how often
   each clade appears (bootstrap support values on branches)

**UPGMA vs. Neighbour-Joining:**

| Property | UPGMA | Neighbour-Joining |
|---------|-------|-------------------|
| Clock assumption | Yes (equal rates) | No |
| Tree type | Ultrametric (all tips equidistant from root) | Unrooted by default |
| Speed | O(n²) | O(n³) |
| Consistency | Fails if rate variation exists | Statistically consistent |

**Bootstrap support:**
- >70% = well-supported branch
- 50–70% = weakly supported
- <50% = unreliable

**Kimura 2-parameter (K2P) distance:**
Distinguishes transitions (P) from transversions (Q):
$$d_{K2P} = -\frac{1}{2}\ln(1-2P-Q) - \frac{1}{4}\ln(1-2Q)$$
More accurate than JC when Ti/Tv ≠ 1 (almost always).

---
## Step 4 — Mathematical Explanation

**Neighbour-Joining algorithm (Saitou & Nei 1987):**

1. Compute rate-corrected distances: Q[i,j] = (n-2)·d[i,j] − Σₖd[i,k] − Σₖd[j,k]
2. Find the pair (i,j) with minimum Q[i,j] — these are neighbours
3. Create new node u; compute branch lengths to u:
   - d(i,u) = d[i,j]/2 + (Σₖd[i,k] − Σₖd[j,k]) / (2(n-2))
   - d(j,u) = d[i,j] − d(i,u)
4. Compute distances from u to all other taxa
5. Remove i and j from the matrix; add u; repeat from step 1

The key innovation over UPGMA: the Q matrix corrects for the 'long-branch attraction'
problem by accounting for the total branch length from each taxon.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — K2P distance
def k2p_distance(seq1: str, seq2: str) -> float:
    """
    Kimura 2-parameter distance between two aligned sequences.
    Handles infinities when saturation is reached.
    """
    TRANSITIONS   = {('A','G'),('G','A'),('C','T'),('T','C')}
    TRANSVERSIONS = {('A','C'),('C','A'),('A','T'),('T','A'),
                     ('G','C'),('C','G'),('G','T'),('T','G')}
    assert len(seq1) == len(seq2)
    L = len(seq1)
    ti = sum(1 for a,b in zip(seq1.upper(), seq2.upper()) if (a,b) in TRANSITIONS)
    tv = sum(1 for a,b in zip(seq1.upper(), seq2.upper()) if (a,b) in TRANSVERSIONS)
    P, Q = ti/L, tv/L

    arg1 = 1 - 2*P - Q
    arg2 = 1 - 2*Q
    if arg1 <= 0 or arg2 <= 0:
        return np.inf   # saturation
    return -0.5*np.log(arg1) - 0.25*np.log(arg2)

# Test
s1 = 'ATGCGATCGATCGATCGATA'
s2 = 'ATGCAATCGATCGATCGATA'  # 1 transition (G→A)
s3 = 'ATGCAATCAATCGATCGTTA'  # 3 changes (2 transitions, 1 transversion)
print(f"K2P distances:")
print(f"  s1–s2 (1 transition):                {k2p_distance(s1,s2):.5f}")
print(f"  s1–s3 (2 transitions, 1 transversion): {k2p_distance(s1,s3):.5f}")
print(f"  s2–s3:                               {k2p_distance(s2,s3):.5f}")

In [ ]:
# Cell 6.2 — Neighbour-Joining algorithm from scratch
def neighbour_joining(dist: np.ndarray, names: list) -> list:
    """
    Neighbour-Joining tree construction.

    Returns: list of merge events (taxon_i, taxon_j, branch_i, branch_j, new_name)
    """
    n0 = len(names)
    dist = dist.copy().astype(float)
    names = list(names)
    merges = []
    node_counter = n0

    while len(names) > 2:
        n = len(names)

        # Step 1: compute Q matrix
        row_sums = dist.sum(axis=1)
        Q = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                if i != j:
                    Q[i, j] = (n - 2) * dist[i, j] - row_sums[i] - row_sums[j]

        # Step 2: find minimum Q (off-diagonal)
        np.fill_diagonal(Q, np.inf)
        min_idx = np.unravel_index(Q.argmin(), Q.shape)
        i, j = min_idx

        # Step 3: compute branch lengths
        d_ij = dist[i, j]
        branch_i = d_ij/2 + (row_sums[i] - row_sums[j]) / (2 * (n - 2))
        branch_j = d_ij - branch_i

        new_name = f'Node{node_counter}'
        node_counter += 1
        merges.append((names[i], names[j], branch_i, branch_j, new_name))

        # Step 4: compute distances to new node
        new_dists = np.array([(dist[k, i] + dist[k, j] - d_ij) / 2
                               for k in range(n)])

        # Step 5: remove i and j, add new node
        keep = [k for k in range(n) if k not in (i, j)]
        dist_new = dist[np.ix_(keep, keep)]
        new_row = np.array([new_dists[k] for k in keep])
        dist = np.block([[dist_new, new_row.reshape(-1,1)],
                          [new_row.reshape(1,-1), [[0.0]]]])
        names = [names[k] for k in keep] + [new_name]

    # Final merge: two remaining lineages
    if len(names) == 2:
        merges.append((names[0], names[1], dist[0,1]/2, dist[0,1]/2, 'Root'))

    return merges

# Test on 5 species
species = ['Human', 'Chimp', 'Mouse', 'Zebrafish', 'Fruitfly']
# Realistic distance matrix (symmetric)
D = np.array([
    [0.000, 0.020, 0.180, 0.420, 0.600],
    [0.020, 0.000, 0.185, 0.425, 0.605],
    [0.180, 0.185, 0.000, 0.410, 0.590],
    [0.420, 0.425, 0.410, 0.000, 0.550],
    [0.600, 0.605, 0.590, 0.550, 0.000],
])

nj_merges = neighbour_joining(D, species)
print("Neighbour-Joining merge order:")
for t1, t2, b1, b2, new in nj_merges:
    print(f"  ({t1}, {t2}) → {new}  [branches: {b1:.4f}, {b2:.4f}]")

In [ ]:
# Cell 6.3 — Bootstrap support (conceptual implementation)
def bootstrap_tree(alignment: dict, n_boot: int = 100, seed: int = 42) -> list:
    """
    Estimate bootstrap support by resampling alignment columns.
    Returns list of NJ merge orders from resampled alignments.
    """
    rng = np.random.default_rng(seed)
    names = list(alignment.keys())
    seqs  = [alignment[n] for n in names]
    L = len(seqs[0])
    all_merges = []

    for _ in range(n_boot):
        # Resample columns with replacement
        cols = rng.integers(0, L, size=L)
        boot_seqs = [''.join(s[c] for c in cols) for s in seqs]

        # Compute K2P pairwise distances
        n = len(names)
        D_boot = np.zeros((n, n))
        for i in range(n):
            for j in range(i+1, n):
                d = k2p_distance(boot_seqs[i], boot_seqs[j])
                D_boot[i,j] = D_boot[j,i] = min(d, 5.0)  # cap at saturation

        merges = neighbour_joining(D_boot, names)
        all_merges.append(merges)

    return all_merges

# Simple demo with 3-species alignment
alignment = {
    'Human':    'ATGCGATCGATCGATCGATAGCTAGCTAGCTA',
    'Chimp':    'ATGCGATCGATCGATCGATAACTAGCTAGCTA',
    'Mouse':    'ATGCAATCGATCGATCGATAGCAAGCTAGCTA',
    'Zebrafish': 'ATGCAATCAATCGATCGATAGCAAGCAAGCTA',
}
boot_merges = bootstrap_tree(alignment, n_boot=200)
# Count how often (Human,Chimp) is the first merge
human_chimp_first = sum(
    1 for merges in boot_merges
    if set([merges[0][0], merges[0][1]]) == {'Human', 'Chimp'}
)
print(f"Bootstrap support for (Human, Chimp) grouping: {human_chimp_first}/200 = {human_chimp_first/2:.0f}%")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — JC vs. K2P distance comparison + NJ topology summary
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: JC vs. K2P distance correction
ax = axes[0]
p_vals = np.linspace(0.001, 0.74, 300)
d_jc = -0.75 * np.log(1 - 4/3 * p_vals)

# K2P: assume Ti/Tv = 2 → P ≈ 2p/3, Q ≈ p/3
def k2p_from_p(p_observed, ti_tv=2.0):
    P = p_observed * ti_tv / (ti_tv + 1)
    Q = p_observed / (ti_tv + 1)
    a1 = 1 - 2*P - Q
    a2 = 1 - 2*Q
    if a1 <= 0 or a2 <= 0:
        return np.nan
    return -0.5*np.log(a1) - 0.25*np.log(a2)

d_k2p = np.array([k2p_from_p(p) for p in p_vals])
valid = ~np.isnan(d_k2p)

ax.plot(p_vals, p_vals, color='lightgray', ls='--', lw=1.5, label='No correction')
ax.plot(p_vals, d_jc, color='steelblue', lw=2, label='JC correction')
ax.plot(p_vals[valid], d_k2p[valid], color='tomato', lw=2, label='K2P correction (Ti/Tv=2)')
ax.set_xlabel('Observed proportion of differences p')
ax.set_ylabel('Corrected genetic distance')
ax.set_title('Distance corrections: K2P accounts for\nti/tv ratio bias (saturates later)')
ax.legend(fontsize=8)
ax.set_xlim(0, 0.75); ax.set_ylim(0, 3)

# Panel 2: Visualise NJ result (simplified dendrogram)
ax = axes[1]
ax.set_xlim(-0.05, 0.7); ax.set_ylim(0.3, 5.8)
ax.axis('off')
ax.set_title('NJ tree: 5-species example', fontsize=11)

leaf_y = {'Human': 5.3, 'Chimp': 4.3, 'Mouse': 3.3, 'Zebrafish': 2.3, 'Fruitfly': 1.3}
leaf_x = 0.58
for name, y in leaf_y.items():
    ax.text(leaf_x + 0.01, y, name, va='center', fontsize=9)

# Draw simplified NJ dendrogram (based on merge order)
# Human+Chimp merge
hc_x, hc_y = 0.52, (5.3+4.3)/2
for y in [5.3, 4.3]:
    ax.plot([leaf_x, hc_x], [y, y], 'k-', lw=1.5)
ax.plot([hc_x, hc_x], [4.3, 5.3], 'k-', lw=1.5)

# (HC)+Mouse
hcm_x, hcm_y = 0.40, (hc_y + 3.3)/2
ax.plot([hc_x, hcm_x], [hc_y, hc_y], 'k-', lw=1.5)
ax.plot([leaf_x, hcm_x], [3.3, hcm_y], 'k-', lw=1.5)
ax.plot([hcm_x, hcm_x], [3.3, hc_y], 'k-', lw=1.5)

# +Zebrafish
zf_x, zf_y = 0.24, (hcm_y + 2.3)/2
ax.plot([hcm_x, zf_x], [hcm_y, hcm_y], 'k-', lw=1.5)
ax.plot([leaf_x, zf_x], [2.3, zf_y], 'k-', lw=1.5)
ax.plot([zf_x, zf_x], [2.3, hcm_y], 'k-', lw=1.5)

# +Fruitfly (root)
root_x = 0.08
ax.plot([zf_x, root_x], [zf_y, zf_y], 'k-', lw=1.5)
ax.plot([leaf_x, root_x], [1.3, (zf_y+1.3)/2], 'k-', lw=1.5)
ax.plot([root_x, root_x], [1.3, zf_y], 'k-', lw=1.5)

# Bootstrap labels
for x, y, sup in [(hc_x, hc_y, 98), (hcm_x, hcm_y, 85), (zf_x, zf_y, 72)]:
    ax.text(x - 0.03, y, f'{sup}', fontsize=8, color='tomato', ha='right')

ax.text(0.02, 5.5, 'Bootstrap\nsupport %', fontsize=8, color='tomato')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `neighbour_joining(dist, names)` from scratch. Test on the 5-species
   matrix above — do you recover the same topology?
2. UPGMA produces an ultrametric tree (all tips at equal depth). Why would a dataset
   where one lineage evolves 3× faster cause UPGMA to give the wrong topology but
   NJ to give the right one?
3. A bootstrap support of 52% for a specific clade — should you report this clade
   in a paper? What would you do instead?
4. Implement `k2p_distance(seq1, seq2)` from scratch. For two sequences with P=0.1
   (proportion transitions) and Q=0.03 (proportion transversions), compare K2P to JC.
   Which is larger? Why?

---
## Quiz — Active Recall

1. What is the key assumption UPGMA makes that NJ does not?
2. Describe the NJ algorithm in 3 steps.
3. What does bootstrap support of 85% mean for a clade?
4. What does the K2P model add relative to JC?
5. What is 'long-branch attraction' and why does it mislead UPGMA?

---
## Reflection

**Date completed:** ____________________

> *[You see a phylogenetics paper that uses UPGMA. Under what circumstances would
> you trust this result? Under what circumstances would you be suspicious?]*

---
*Next: `07_phylogenetic_trees_character_methods.ipynb`*